# Understanding subcellular localisation of E-Cadherin
- Some papers have shown that increased nuclear E-cadherin localisation is inversely correlated to decreased membrane bound E-cadherin localisation
- Instead of just looking at colocalisation, divide the cell into 3 parts:
    - Membrane
    - Nucleus
    - Cytoplasm (acinus - membrane - nuclei)
- Easier to do this for full acini?
    - Then there's no issue attributing membrane E-cadherin to either one of the neighbouring cells

- Just use the membrane segmentation as the membrane marker, without using the identified nuclei as seed points to expand out cytoplasmic areas
    - This introduces more error and sometimes the exact border of the identified cells doesn't align with the true membrane

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import math
import os
import pathlib
from joblib import Parallel, delayed

from skimage import exposure, measure, util
from skimage.filters import threshold_otsu, gaussian
from skimage.segmentation import clear_border, find_boundaries, watershed, expand_labels
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import remove_small_holes, remove_small_objects, closing, opening, ball, skeletonize, thin, dilation, erosion, medial_axis
from skimage.transform import rescale
from skimage.feature import peak_local_max

from scipy.ndimage import distance_transform_edt
from scipy import ndimage as ndi

from tifffile import imread
import tifffile
from typing import Tuple, List
from itertools import combinations
import textwrap

import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from textwrap import wrap
from matplotlib.patches import Patch
from matplotlib.collections import PatchCollection
from itertools import groupby
import napari
import re
import warnings
warnings.filterwarnings("ignore")
mycol = ["red", "darkorange", "yellow", "limegreen", "dodgerblue", "darkviolet", "deeppink" ]
from sklearn.linear_model import LinearRegression
import colorsys
from PIL import ImageColor 

In [2]:
import contextlib
import joblib
from tqdm import tqdm

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib to report into tqdm progress bar given as argument"""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [3]:
def create_palette(n, base_color_hex):
    r,g,b = ImageColor.getcolor(base_color_hex, "RGB")[0], ImageColor.getcolor(base_color_hex, "RGB")[1], ImageColor.getcolor(base_color_hex, "RGB")[2]
    r=r/255
    g=g/255
    b=b/255
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    lightness_adjustments = np.linspace(0.7, 1.3, n)  # Creating a range of values
    # Create the palette with adjusted lightness while keeping hue and saturation constant
    palette = [colorsys.hls_to_rgb(h, min(max(l * adj, 0), 1), s) for adj in lightness_adjustments]
    return palette

In [4]:
def pixel_size(tif_path):
    with tifffile.TiffFile(tif_path) as tif:
        tif_tags = {}
        for tag in tif.pages[0].tags.values():
            name, value = tag.name, tag.value
            tif_tags[name] = value

        x_pixel_size_um = 1/((tif_tags["XResolution"])[0]/(tif_tags["XResolution"][1]))
        y_pixel_size_um = 1/((tif_tags["YResolution"])[0]/(tif_tags["YResolution"][1]))
        try:
            z_pixel_size_um = float(str(tif_tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
        except:
            z_pixel_size_um = (float(str(tif_tags["ImageDescription"]).split("spacing=")[1].split("loop")[0]))
       
    original_spacing = [x_pixel_size_um,y_pixel_size_um,z_pixel_size_um]
    return original_spacing

In [5]:
def add_image_details(df, filename, flag):
    df["filename"] = filename
    df["flag"] = flag
    if ("well1" in filename):
        df["well"] = 1
    else:
        df["well"] = 2
    # Day #        
    if "d0" in filename:
        df["day"] =  0
    elif "d1" in filename:
        df["day"] =  1
    elif "d3" in filename:
        df["day"] =  3
    else:
        df["day"] =  7
    # Stiffness
    if 'soft' in filename:
        df["condition"] =  'soft'
    elif 'stiff' in filename:
        df["condition"] =  'stiff'
    else:
        df["condition"] =  'blank'
    df['image_type'] = df["condition"].astype(str) + ", d" + df["day"].astype(str)

    return df

In [6]:
def segment_and_quantify(i, js, ks, image_paths, membrane_paths, nuclei_paths, poi_channel = 1,  membrane_channel= 2, dapi_channel = 0, to_plot = False):
    filename = os.path.basename(image_paths[i]).replace("_", "").lower()
    try:
    ##############RESCALE IMAGES
        flag = "None"
        dapi_mask = imread(nuclei_paths[ks[i]])
        membrane_mask = imread(membrane_paths[js[i]])
        poi_image = imread(image_paths[i])[:,poi_channel,:,:]
        caax_image = imread(image_paths[i])[:,membrane_channel,:,:]
        dapi_image = imread(image_paths[i])[:,dapi_channel,:,:]

        original_spacing = pixel_size(nuclei_paths[i])
        scale_change = original_spacing[2] / original_spacing[0]

        rescaled_dapi_mask = rescale(scale = (0.25*scale_change, 0.25, 0.25), image = dapi_mask, anti_aliasing= False)
        rescaled_membrane_mask = rescale(scale = (0.25*scale_change, 0.25, 0.25), image = membrane_mask, anti_aliasing= False)
        rescaled_poi_image = rescale(scale = (0.25*scale_change, 0.25, 0.25), image = poi_image, anti_aliasing= False)
        rescaled_caax_image = rescale(scale = (0.25*scale_change, 0.25, 0.25), image = caax_image, anti_aliasing= False)
        rescaled_dapi_image = rescale(scale = (0.25*scale_change, 0.25, 0.25), image = dapi_image, anti_aliasing= False)
        rescaled_acinus_image = rescale(scale = (0.25*scale_change, 0.25, 0.25), image = caax_image + dapi_image, anti_aliasing= False)
    #######################################Identify acinus and set evertyhing outside it to 0#######################################
        """
        FIND THE SHAPE OF THE WHOLE ACINUS AND GET SET ANYTHING OUTSIDE TO 0
        Check for acinar sphericity -> low sphericity could mean that 2 neighbouring acini have been imaged. Attempt to separate these with heavy erosion if so
        """
        clipped = rescaled_acinus_image.clip(min =np.quantile(rescaled_acinus_image, 0.1), max = np.quantile(rescaled_acinus_image, 0.85))
        acinus_smoothed = gaussian(clipped, sigma = 4)
        thresh = threshold_otsu(acinus_smoothed)
        binary = acinus_smoothed > thresh
        binary = remove_small_holes(binary, area_threshold = 100000)
        binary = remove_small_objects(binary, min_size = 10000)
        labelled_image = label(binary)
        table = regionprops_table(labelled_image, properties=('label', "area"),) #only keep largest acinus
        condition = (table['area'] >= (table["area"]).max())
        input_labels = table['label']
        output_labels = input_labels * condition
        filtered_labelled_acinus = util.map_array(labelled_image, input_labels, output_labels)
        table = regionprops_table(filtered_labelled_acinus, properties=('label', 'inertia_tensor_eigvals'),) #test sphericity of largest identified acinus
        condition = table['inertia_tensor_eigvals-2']/table['inertia_tensor_eigvals-0'] >= (0.55)
        if condition == False:
            flag = "multiple_acini_split"
            acinus_smoothed = gaussian(clipped, sigma = 1)
            thresh = threshold_otsu(acinus_smoothed)
            binary = acinus_smoothed > thresh
            binary = remove_small_holes(binary, area_threshold = 100000)
            binary = remove_small_objects(binary, min_size = 10000)
            binary = erosion(binary, ball(8))
            labelled_image = label(binary)
            table = regionprops_table(labelled_image, properties=('label', "area"),)
            condition = (table['area'] >= (table["area"]).max())
            input_labels = table['label']
            output_labels = input_labels * condition
            filtered_labelled_acinus = util.map_array(labelled_image, input_labels, output_labels)
            filtered_labelled_acinus = expand_labels(filtered_labelled_acinus, distance=8)
        cleaned_acinus_mask = filtered_labelled_acinus * (filtered_labelled_acinus>0)
    ####################################################################################################################################

        rescaled_membrane_mask = rescaled_membrane_mask * cleaned_acinus_mask
        rescaled_dapi_mask = rescaled_dapi_mask * cleaned_acinus_mask

    ####################PROCESS THE MEMBRANE TO THIN AND CLEAN####################
        cleaned_membrane_mask = gaussian(rescaled_membrane_mask, sigma = 0.5) # Apply some smoothing to fill any gaps
        thresh = threshold_otsu(cleaned_membrane_mask)
        cleaned_membrane_mask = cleaned_membrane_mask > thresh
        
    #####################CLEAN UP THE NUCLEI AND LABEL####################
        thresh = threshold_otsu(rescaled_dapi_mask)
        cleaned_dapi_mask = rescaled_dapi_mask > thresh
        cleaned_dapi_mask = remove_small_objects(cleaned_dapi_mask, min_size = 60)
        cleaned_dapi_mask = remove_small_holes(cleaned_dapi_mask, area_threshold = 60)

        cleaned_dapi_mask = cleaned_dapi_mask * (util.invert(cleaned_membrane_mask)) # Don't allow nuclei to overlap membrane
        cleaned_cyto_mask = util.invert(cleaned_membrane_mask) * cleaned_acinus_mask * util.invert(cleaned_dapi_mask) # Cyto = acinus - membrane - dapi

    #####################QUANTIFY POI IN ALL SUBCELLULAR COMPARTMENTS####################
        poi_membrane = rescaled_poi_image*(cleaned_membrane_mask)
        poi_nuclei = rescaled_poi_image*cleaned_dapi_mask
        poi_cyto = rescaled_poi_image * (cleaned_cyto_mask)

        poi_acinus = poi_membrane + poi_nuclei + poi_cyto


        nuclear_poi_fraction = poi_nuclei.sum()/poi_acinus.sum()
        membrane_poi_fraction = poi_membrane.sum()/poi_acinus.sum()
        cyto_poi_fraction = poi_cyto.sum()/poi_acinus.sum()

        output = pd.DataFrame({"filename":filename, "nuclear_fraction": [nuclear_poi_fraction], "membrane_fraction":membrane_poi_fraction, "cyto_fraction": cyto_poi_fraction, "total":nuclear_poi_fraction+membrane_poi_fraction+cyto_poi_fraction})
        add_image_details(output, filename, flag)
        if to_plot == True:
            return output, cleaned_dapi_mask, cleaned_cyto_mask, cleaned_membrane_mask, rescaled_caax_image, rescaled_dapi_image, rescaled_acinus_image, rescaled_poi_image
        return output
    except:
        output = pd.DataFrame({"filename":filename, "nuclear_fraction": ["FAIL"], "membrane_fraction":["FAIL"], "cyto_fraction": ["FAIL"], "total":["FAIL"]})


In [7]:
# images_root = "E:\\PROTEIN_COLOCALISATION_IMAGES\\Deconvolved_Images\\FINAL_COL_BETA_CAT\\tifs\\using"
# dapi_root = "E:\\PROTEIN_COLOCALISATION_IMAGES\\Deconvolved_Images\\FINAL_COL_BETA_CAT\\tifs\\split_using\\COMBINED_binary_dapi"
# membrane_root = "E:\\PROTEIN_COLOCALISATION_IMAGES\\Deconvolved_Images\\FINAL_COL_BETA_CAT\\tifs\\split_using\\COMBINED_binary_membrane"
main_root = "E:\\PROTEIN_COLOCALISATION_IMAGES\\Misp_Alphacat"
membrane_root = main_root + "\\segmentations\\combined_binary_membrane"
dapi_root = main_root + "\\segmentations\\combined_binary_dapi"
images_root = main_root + "\\final_tifs"

membrane_paths = list(pathlib.Path(membrane_root).glob("**/*.tif"))
nuclei_paths = list(pathlib.Path(dapi_root).glob("**/*.tif"))
image_paths = list(pathlib.Path(images_root).glob("**/*.tif"))
membrane_paths.sort()
nuclei_paths.sort()
image_paths.sort()
print(len(membrane_paths), len(nuclei_paths), len(image_paths))

filenames_to_match = [os.path.basename(image_paths[i]).lower().replace("_", "").split(".")[0] for i in range(len(image_paths))]
membrane_filenames_to_match = [("".join(os.path.basename(membrane_paths[i]).lower().split("_")[1:-2])) for i in range(len(image_paths))]
dapi_filenames_to_match = [("".join(os.path.basename(nuclei_paths[i]).lower().split("_")[1:-2])) for i in range(len(image_paths))]
js = []
ks = []
for i in range(len(image_paths)):
    j = membrane_filenames_to_match.index(filenames_to_match[i])
    js.append(j)
    k = dapi_filenames_to_match.index(filenames_to_match[i])
    ks.append(k)

60 60 60


In [8]:
i=15
output, cleaned_dapi_mask, cleaned_cyto_mask, cleaned_membrane_mask, rescaled_caax_image, rescaled_dapi_image, rescaled_acinus_image ,rescaled_poi_image = segment_and_quantify(i, js, ks, image_paths, membrane_paths, nuclei_paths, to_plot = True)
output

,filename,nuclear_fraction,membrane_fraction,cyto_fraction,total,flag,well,day,condition,image_type
0,mispalphacatd0well22p0zoom0062popalphacat.tif,0.450744,0.365643,0.183614,1.0,None,2,0,blank,"blank, d0"


In [9]:
palette_limegreen = create_palette(1,"#32CD32")
palette_dodgerblue = create_palette(1, "#1E90FF")
palette_red = create_palette(1, "#FF0000")
combined_palette = palette_dodgerblue + palette_red + palette_limegreen

custom_colormap = {0: np.array([0., 0., 0., 0.]),} #black backgroun
for index, element in enumerate(combined_palette):
    custom_colormap[index+1] = element

In [10]:
viewer= napari.Viewer()
viewer.add_image(rescaled_poi_image)
viewer.add_image(rescaled_dapi_image)
viewer.add_image(rescaled_caax_image)
viewer.add_image(rescaled_acinus_image)
viewer.add_labels(cleaned_dapi_mask, color= custom_colormap)
viewer.add_labels(2*cleaned_cyto_mask, color= custom_colormap)
viewer.add_labels(3*cleaned_membrane_mask, color= custom_colormap)

<Labels layer 'Labels [1]' at 0x25aa83f12d0>

# Full Run

In [11]:
main_root = "E:\\PROTEIN_COLOCALISATION_IMAGES\\Misp_Alphacat"

main_root = "E:\\PROTEIN_COLOCALISATION_IMAGES\\Misp_Alphacat"
membrane_root = main_root + "\\segmentations\\combined_binary_membrane"
dapi_root = main_root + "\\segmentations\\combined_binary_dapi"
images_root = main_root + "\\final_tifs"

membrane_paths = list(pathlib.Path(membrane_root).glob("**/*.tif"))
nuclei_paths = list(pathlib.Path(dapi_root).glob("**/*.tif"))
image_paths = list(pathlib.Path(images_root).glob("**/*.tif"))
membrane_paths.sort()
nuclei_paths.sort()
image_paths.sort()
print(len(membrane_paths), len(nuclei_paths), len(image_paths))

filenames_to_match = [os.path.basename(image_paths[i]).lower().replace("_", "").split(".")[0] for i in range(len(image_paths))]
membrane_filenames_to_match = [("".join(os.path.basename(membrane_paths[i]).lower().split("_")[1:-2])) for i in range(len(image_paths))]
dapi_filenames_to_match = [("".join(os.path.basename(nuclei_paths[i]).lower().split("_")[1:-2])) for i in range(len(image_paths))]
js = []
ks = []
for i in range(len(image_paths)):
    j = membrane_filenames_to_match.index(filenames_to_match[i])
    js.append(j)
    k = dapi_filenames_to_match.index(filenames_to_match[i])
    ks.append(k)

with tqdm_joblib(tqdm(desc="Image Analysis", total=len(image_paths))) as progress_bar:
    output = Parallel(n_jobs=3)(delayed(segment_and_quantify)(i, js, ks, image_paths, membrane_paths, nuclei_paths) for i in range(len(image_paths)))
output = pd.concat(output)
output.to_csv("alphacat_subcellular_localisation.csv")


60 60 60


Image Analysis: 100%|██████████| 60/60 [18:49<00:00, 18.82s/it]


In [12]:
main_root_ecad = "E:\\PROTEIN_COLOCALISATION_IMAGES\\Deconvolved_Images\\ECad_C3_DECONVOLVED\\tifs"
main_root_col = "E:\\PROTEIN_COLOCALISATION_IMAGES\\Deconvolved_Images\\FINAL_COL_BETA_CAT\\tifs"
main_roots = [main_root_ecad,  main_root_col]
proteins = ["E_cadherin", "Beta_Catenin"]

for my_count, main_root in enumerate(main_roots):
    membrane_root = main_root + "\\segmentations\\combined_binary_membrane"
    dapi_root = main_root + "\\segmentations\\combined_binary_dapi"
    images_root = main_root + "\\final_tifs"

    membrane_paths = list(pathlib.Path(membrane_root).glob("**/*.tif"))
    nuclei_paths = list(pathlib.Path(dapi_root).glob("**/*.tif"))
    image_paths = list(pathlib.Path(images_root).glob("**/*.tif"))
    membrane_paths.sort()
    nuclei_paths.sort()
    image_paths.sort()

    poi = proteins[my_count]

    print(len(membrane_paths), len(nuclei_paths))
    if len(membrane_paths) != len(nuclei_paths):
        print("CHECK DIRECTORY!")
        print(len(membrane_paths), len(nuclei_paths))

    #####FOR EVERYTHING BUT LAMPAX
    filenames_to_match = [os.path.basename(image_paths[i]).lower().replace("_", "").split(".")[0] for i in range(len(image_paths))]
    membrane_filenames_to_match = [("".join(os.path.basename(membrane_paths[i]).lower().split("_")[1:-2])) for i in range(len(image_paths))]
    dapi_filenames_to_match = [("".join(os.path.basename(nuclei_paths[i]).lower().split("_")[1:-2])) for i in range(len(image_paths))]
    js = []
    ks = []
    for i in range(len(image_paths)):
        j = membrane_filenames_to_match.index(filenames_to_match[i])
        js.append(j)
        k = dapi_filenames_to_match.index(filenames_to_match[i])
        ks.append(k)

    with tqdm_joblib(tqdm(desc="Image Analysis", total=len(image_paths))) as progress_bar:
        output = Parallel(n_jobs=3)(delayed(segment_and_quantify)(i, js, ks, image_paths, membrane_paths, nuclei_paths) for i in range(len(image_paths)))
    output = pd.concat(output)
    output.to_csv("{}_subcellular_localisation.csv".format(poi))


140 140


Image Analysis:   0%|          | 0/140 [00:00<?, ?it/s]

Image Analysis: 100%|██████████| 140/140 [36:48<00:00, 15.77s/it]


140 140


Image Analysis: 100%|██████████| 140/140 [41:38<00:00, 17.84s/it]
